# NLP Assignment 1: Colab build (full pipeline)

**Covers**
- Step 0: raw corpus verification, cleaned corpus, cleaning verification, vocabulary and token array.
- Q1.1 to Q1.8: n-gram model, probabilities, stopwords, sentence scores, generation (stage 10).
- Q2: held-out perplexity, seed 42 (primary) and seed 7 (replication) (stage 11).
- Q3: real versus corrupted sentence detection, with UNK baseline (stage 12).
- Q4: next-word prediction, Top-1, Top-5, MRR@5 (stage 13).
- Presentation artifacts: export bundle and checksums.

**Where each result is printed**
- Raw corpus hash: the "Fetch raw corpus" cell.
- Step 0 shares: the "Gate checks" cell.
- Q1.3 sums: the "Gate checks" cell.
- Q1.4 to Q1.8 tables: the "Results summary" cell.
- Q2, Q3, Q4 metrics: the "Gate checks" cell (full JSON) and the "Results summary" cell.

**Rules**
- Heavy stages run only in this notebook or on Kaggle (AGENTS B13). The PC runs tests, `check_download.py`, and the app.
- Before running, choose *Runtime → Change runtime type → High-RAM* if your account offers it. The memory cell prints the limit you get.
- Re-running the notebook resumes. Each stage skips itself if its manifest entry still matches. Outputs are copied to Google Drive after every stage, so a disconnect does not lose them.
- Every number is printed by code. The markdown after each cell only describes what the printed output shows (AGENTS B8).


## 1. Runtime report


In [ ]:
import os, sys, platform, shutil

def read_first(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None

def physical_ram_bytes():
    try:
        return os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
    except (ValueError, OSError):
        return None

try:
    import google.colab  # noqa: F401
    on_colab = True
except ImportError:
    on_colab = False

phys = physical_ram_bytes()
cgroup = read_first("/sys/fs/cgroup/memory.max") or read_first("/sys/fs/cgroup/memory/memory.limit_in_bytes")
print("Running on Colab       :", on_colab)
print("Physical RAM (GiB)     :", round(phys / 2**30, 2) if phys else "unknown")
print("cgroup memory limit    :", cgroup)
print("CPUs                   :", os.cpu_count())
print("Disk free, /content GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
print("Python                 :", platform.python_version())


Result: the physical RAM and cgroup limit above are the limits this runtime enforces. If they are lower than you expected, the runtime is the standard one. Change to High-RAM under *Runtime → Change runtime type* and run this cell again.


## 2. Dependencies


In [ ]:
!pip install -q gdown
import numpy, sklearn, pandas
print("numpy        :", numpy.__version__)
print("scikit-learn :", sklearn.__version__)
print("pandas       :", pandas.__version__)

PINNED_NUMPY = "2.1.3"
assert numpy.__version__ == PINNED_NUMPY, (
    f"numpy {numpy.__version__} is installed, but the split is pinned to {PINNED_NUMPY}. "
    f"Run !pip install -q numpy=={PINNED_NUMPY}, restart the session, and run the notebook from the top."
)
print("numpy pin ok:", PINNED_NUMPY)


Result: the three library versions above are recorded in `README.md` (AGENTS B4).


## 3. Clone or update the repository

The clone never deletes an existing folder. If the repository is already present, it updates it, so `data/` and the manifest survive a re-run.


In [ ]:
import os, subprocess
from pathlib import Path

REPO = "/content/NLP_Assignment1"
REPO_URL = "https://github.com/KenaKaye/Assignment-1.git"
BRANCH = "full-corpus"

os.chdir("/content")
if not Path(REPO, ".git").is_dir():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO)
print("Commit used by this run:")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


Result: the commit printed above is the code version this run used. Record it in `README.md`.


## 4. Memory budget

The counting stages choose their method from this budget. It comes from the runtime's limit, not from a fixed number.


In [ ]:
import sys
REPO = "/content/NLP_Assignment1"
sys.path.insert(0, REPO)
from scripts.common import memory_budget_mb, REPO_ROOT
print("Repository root :", REPO_ROOT)
print("Memory budget MB:", round(memory_budget_mb(), 0))


Result: the budget above is the limit used for every counting decision in this run.


## 5. Connect Google Drive

The Drive account must be the one that holds `GPAC.txt.gz` (and any earlier backup).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Result: the cell finished without an error, so Drive is mounted at `/content/drive/MyDrive`.


## 6. Restore the Drive backup, if one exists

Restored files are only used if each stage's manifest still matches them. The stages check this themselves.


In [ ]:
import os, shutil
from pathlib import Path

REPO_ROOT = Path("/content/NLP_Assignment1")
BACKUP = Path("/content/drive/MyDrive/NLP_Assignment1_backup")

if BACKUP.is_dir():
    restored = 0
    for src in BACKUP.rglob("*"):
        if not src.is_file():
            continue
        dst = REPO_ROOT / src.relative_to(BACKUP)
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        restored += 1
    print(f"Restored {restored} files from {BACKUP}")
else:
    print("No Drive backup found. Starting with an empty data folder.")


Result: the number of files restored is printed above. Nothing is trusted until the stage checks run.


## 7. Fetch the raw corpus and verify it (Step 0)

The raw file is decompressed from `GPAC.txt.gz` in Drive. The hash is checked against the authoritative SHA-256 in the plan. If a copy already exists and matches, it is not decompressed again.


In [ ]:
import gzip, shutil, hashlib, glob, os
from pathlib import Path

EXPECTED = "977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa"
RAW = Path("/content/NLP_Assignment1/data/GPAC.txt")

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW.parent.mkdir(parents=True, exist_ok=True)
if RAW.exists() and sha256_of(RAW) == EXPECTED:
    print("Raw corpus already present and matching.")
else:
    candidates = glob.glob("/content/drive/MyDrive/GPAC.txt.gz") + glob.glob("/content/drive/MyDrive/*/GPAC.txt.gz")
    if not candidates:
        raise SystemExit("GPAC.txt.gz not found in My Drive or one folder deep. Upload it, then run this cell again.")
    SRC = candidates[0]
    print("Decompressing:", SRC)
    with gzip.open(SRC, "rb") as src, open(RAW, "wb") as dst:
        shutil.copyfileobj(src, dst)

actual = sha256_of(RAW)
print("ACTUAL   SHA-256:", actual)
print("EXPECTED SHA-256:", EXPECTED)
assert actual == EXPECTED, "Raw corpus hash mismatch. This is not the authoritative corpus. Stop here."
print("Raw corpus SHA-256 matches the authoritative value.")


Result: the two hashes printed above are identical, and the assertion passed. If the assertion failed, stop: the file is not the authoritative corpus.


## 8. Reset stale outputs, if any

The cleaned corpus from an older run has no sentence-final punctuation. Its stages cannot be reused, so this cell removes stages 02 to 13. The raw file, `sentence_assessment.tsv`, the stage 01 manifest entry, and the run log are kept.


In [ ]:
import json, shutil
from pathlib import Path

REPO = Path("/content/NLP_Assignment1")
os.chdir(REPO)
DATA = REPO / "data"
RESULTS = REPO / "results"
PUNCT = ("።", "?", "!")
KEEP_STAGE = "01_assess_raw"

def cleaned_is_stale(path, sample=2000, threshold=0.99):
    if not path.exists():
        print("No cleaned corpus yet. Nothing to reset.")
        return False
    total = ok = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.rstrip("\n")
            if not s:
                continue
            total += 1
            ok += s.endswith(PUNCT)
            if total >= sample:
                break
    frac = ok / total if total else 0.0
    print(f"Cleaned lines ending in sentence punctuation (first {total}): {frac:.4f}")
    return frac < threshold

if not cleaned_is_stale(DATA / "GPAC_cleaned.txt"):
    print("Cleaned corpus is current. No reset.")
else:
    print("Stale cleaned corpus. Resetting stages 02 to 13.")
    manifest_path = DATA / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    for stage in list(manifest):
        if stage == KEEP_STAGE:
            continue
        for rel in manifest[stage].get("outputs", {}):
            p = REPO / rel
            if p.is_file():
                p.unlink()
        manifest.pop(stage)
    for pattern in ["GPAC_cleaned.txt", "*.npy", "*.tsv", "split_indices_*.json", "lut_*.npy"]:
        for p in DATA.glob(pattern):
            if p.name != "sentence_assessment.tsv" and p.name != "GPAC.txt":
                p.unlink()
    shutil.rmtree(DATA / "counts", ignore_errors=True)
    for p in RESULTS.glob("*"):
        if p.is_file() and p.name != "run_log.jsonl":
            p.unlink()
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    cks_path = DATA / "checksums.json"
    if cks_path.exists():
        cks = json.loads(cks_path.read_text(encoding="utf-8"))
        cks = {k: v for k, v in cks.items() if (REPO / k).exists()}
        cks_path.write_text(json.dumps(cks, indent=2, sort_keys=True), encoding="utf-8")
    print("Reset complete.")


Result: the fraction printed above is near 1 for a current cleaned corpus. If it is below 0.99, the reset ran. Either way, the message above says which path was taken.


## 9. Run the pipeline (stages 01 to 13)

Each stage runs as a separate process and prints its own output. A failed stage stops the run and names itself. Stages with a valid manifest entry are skipped. After each stage, the data and results are copied to Google Drive.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = "/content/NLP_Assignment1"
os.chdir(REPO)
os.makedirs("results", exist_ok=True)

STAGES = [
    ["scripts/01_assess_raw.py"],
    ["scripts/02_build_cleaned.py"],
    ["scripts/03_verify_cleaned.py"],
    ["scripts/04_build_vocab_tokens.py"],
    ["scripts/06_split.py"],
    ["scripts/07_count_full.py"],
    ["scripts/08_count_train.py", "--seed", "42"],
    ["scripts/08_count_train.py", "--seed", "7"],
    ["scripts/10_q1_results.py"],
    ["scripts/11_q2_perplexity.py"],
    ["scripts/12_q3_extrinsic.py"],
    ["scripts/13_q4_eval.py"],
]

REPO_ROOT = Path(REPO)
BACKUP = Path("/content/drive/MyDrive/NLP_Assignment1_backup")

def backup_to_drive():
    if not Path("/content/drive/MyDrive").is_dir():
        print("Drive not mounted. Backup skipped.")
        return
    copied = 0
    sources = [p for p in (REPO_ROOT / "data").rglob("*") if p.is_file()]
    sources += [p for p in (REPO_ROOT / "results").rglob("*") if p.is_file()]
    for src in sources:
        if src.name == "GPAC.txt":
            continue
        dst = BACKUP / src.relative_to(REPO_ROOT)
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1
    print(f"Backup to Drive: {copied} files copied")

times_path = Path("results/colab_stage_times.json")
times = json.loads(times_path.read_text()) if times_path.exists() else {}

for args in STAGES:
    name = " ".join(args)
    print(f"\n==================== {name} ====================", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable, "-u", *args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    dur = round(time.time() - t0, 1)
    times[name] = dur
    times_path.write_text(json.dumps(times, indent=2), encoding="utf-8")
    print(f"== {name}: exit code {rc}, elapsed {dur} s ==", flush=True)
    if rc != 0:
        raise RuntimeError(
            f"Stage failed: {name} (exit code {rc}). Fix the cause and run this cell again. "
            "Finished stages will be skipped."
        )
    backup_to_drive()

print("\nPipeline finished. Stage times (s):")
for k, v in times.items():
    print(f"  {k}: {v}")


Result: every stage printed its exit code and elapsed time above. Each stage's own numbers are printed in its output block. Stage times are saved to `results/colab_stage_times.json` for comparison with the plan's estimates.


## 10. Gate checks

Each check prints its value and asserts against the benchmark or the rule. Nothing is marked as passed unless its assertion ran.


In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np

os.chdir("/content/NLP_Assignment1")

def load_json(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing {path}. The stage that writes it did not complete.")
    return json.loads(p.read_text(encoding="utf-8"))

print("--- Unit tests (light suite, also run on the PC) ---")
rc = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"]).returncode
assert rc == 0, "Unit tests failed. Do not use any result from this run."
print("Unit tests exit code:", rc)

print("\n--- Step 0: cleaning and verification ---")
v = load_json("data/cleaned_verification_record.json")
print("Verification record fields:", sorted(v.keys()))
letter = v["ethiopic_letter_share_pct"]
punct = v["ethiopic_with_punctuation_share_pct"]
print(f"Ethiopic letter share     : {letter:.2f}%   (benchmark: > 80%)")
print(f"Ethiopic plus punctuation : {punct:.2f}%")
assert letter >= 80.0, "Ethiopic letter share below 80%"

print("\n--- Q1.3: counts and probability sums (AGENTS B6: within 1e-9) ---")
N = len(np.load("data/tokens.npy", mmap_mode="r"))
print(f"Token positions N: {N:,}")
for n in range(1, 5):
    c = np.load(f"data/counts/full/order{n}_counts.npy")
    total = int(c.sum())
    p_sum = total / N
    print(f"  order {n}: sum(counts) = {total:,}, sum(probabilities) = {p_sum:.12f}")
    assert total == N, f"order {n}: counts do not sum to N"
    assert abs(p_sum - 1.0) <= 1e-9, f"order {n}: probabilities do not sum to 1 within 1e-9"

print("\n--- Q1.8: verbatim copies, full corpus ---")
gen = load_json("results/q1_generation.json")
print("Keys:", list(gen.keys()))
for order_key, sents in gen.items():
    flags = [bool(s["is_full_verbatim_copy"]) for s in sents]
    print(f"  {order_key}: {sum(flags)} of {len(flags)} generated sentences are verbatim corpus lines")

print("\n--- Q2: held-out perplexity (both seeds) ---")
q2 = load_json("results/q2_perplexity.json")
print("Top-level keys:", list(q2.keys()))
print(json.dumps(q2, indent=1, ensure_ascii=False))

print("\n--- Q3: extrinsic metrics ---")
q3 = load_json("results/q3_metrics.json")
print("Top-level keys:", list(q3.keys()))
print(json.dumps(q3, indent=1, ensure_ascii=False))

print("\n--- Q4: predictive text ---")
q4 = load_json("results/q4_eval.json")
print("Top-level keys:", list(q4.keys()))
print(json.dumps(q4, indent=1, ensure_ascii=False))


Result: the unit tests, the Step 0 shares, the Q1.3 sums, the Q1.8 verbatim counts, and the full Q2, Q3, and Q4 result files are printed above. The first failed assertion stops the cell. Describe a benchmark as met only when the printed `meets_benchmarks` value is true, and a classifier as usable only when its printed `usable` value is true.


## 11. Results summary (Q1 tables)


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
for n in range(1, 5):
    print(f"\n=== Q1.3 top 10, n = {n} ===")
    print(pd.read_csv(f"results/q1_top10_n{n}.csv").to_string(index=False))

print("\n=== Q1.4 conditional probabilities ===")
print(pd.read_csv("results/q1_bigram_conditional.csv").to_string(index=False))

for n in range(1, 5):
    print(f"\n=== Q1.5 top 10 after stopword removal, n = {n} ===")
    print(pd.read_csv(f"results/q1_stopword_top10_n{n}.csv").to_string(index=False))

print("\n=== Q1.7 sentence log-probabilities ===")
print(pd.read_csv("results/q1_sentence_probs.csv").to_string(index=False))


Result: the tables above are the Q1.3, Q1.4, Q1.5, and Q1.7 outputs as the code wrote them. Q1.6 word-cloud inputs are in `results/q1_wordcloud_*.csv` and are rendered on the PC with Noto Sans Ethiopic (AGENTS B7).


## 12. Export the canonical artifacts

The list comes from `scripts/artifacts.py`. The export fails if any listed artifact is missing. The zip is created **after** every file is copied, so it contains the checksum, manifest, and log files too.


In [ ]:
import json, os, shutil, sys
from pathlib import Path

REPO = Path("/content/NLP_Assignment1")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from scripts.artifacts import write_artifact_list

write_artifact_list()
artifacts = json.loads(Path("data/artifact_list.json").read_text(encoding="utf-8"))

EXPORT = Path("/content/export")
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)

missing, total = [], 0
for rel in artifacts:
    src = REPO / rel
    if not src.is_file():
        missing.append(rel)
        continue
    dst = EXPORT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    total += dst.stat().st_size

for extra in ["data/checksums.json", "data/manifest.json", "data/artifact_list.json",
              "results/run_log.jsonl", "results/colab_stage_times.json"]:
    src = REPO / extra
    if src.is_file():
        dst = EXPORT / extra
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        total += dst.stat().st_size

print(f"Listed artifacts : {len(artifacts)}")
print(f"Copied           : {len(artifacts) - len(missing)}")
print(f"Missing          : {missing}")
print(f"Total size       : {total / 2**30:.2f} GiB")
assert not missing, "Canonical artifacts are missing. Fix them before exporting."

zip_path = shutil.make_archive("/content/nlp_assignment1_artifacts", "zip", EXPORT)
print("Created", zip_path, round(os.path.getsize(zip_path) / 2**30, 2), "GiB")


Result: the listed, copied, and missing counts and the zip size are printed above. If anything is missing, the cell stops before the zip is made.


## 13. Copy the zip to Google Drive (optional, recommended)

Copying inside Google's network is fast. Downloading the zip from Drive to your PC uses your own download speed.


In [ ]:
import shutil
from pathlib import Path

ZIP = Path("/content/nlp_assignment1_artifacts.zip")
DEST = Path("/content/drive/MyDrive/NLP_Assignment1_export")
if Path("/content/drive/MyDrive").is_dir():
    DEST.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP, DEST / ZIP.name)
    print("Copied to", DEST / ZIP.name)
else:
    print("Drive not mounted. Download the zip from the Colab files pane instead.")


## 14. Next steps on the PC

1. Download `nlp_assignment1_artifacts.zip` from Drive, or from the Colab files pane.
2. Extract it into the repository root. The `data/` and `results/` folders merge with the existing ones.
3. Run `python scripts/check_download.py` (or `uv run scripts/check_download.py`). Every checksum and the raw corpus hash must match. Do not run anything else until this passes (AGENTS B13).
4. Then run `python -m pytest tests/`, and launch the CLI with `python app/cli.py`.
